In [3]:
from transformers import EsmTokenizer, EsmForMaskedLM

model_name = 'facebook/esm1v_t33_650M_UR90S_1'
tokenizer = EsmTokenizer.from_pretrained(model_name)
model = EsmForMaskedLM.from_pretrained(model_name)

tokenizer_config.json:   0%|          | 0.00/40.0 [00:00<?, ?B/s]

d:\Projects\iGEM-stuff\venv\Lib\site-packages\huggingface_hub\file_download.py:143: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\dante\.cache\huggingface\hub\models--facebook--esm1v_t33_650M_UR90S_1. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


vocab.txt:   0%|          | 0.00/93.0 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/785 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/2.61G [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/2.61G [00:00<?, ?B/s]

Some weights of EsmForMaskedLM were not initialized from the model checkpoint at facebook/esm1v_t33_650M_UR90S_1 and are newly initialized: ['esm.contact_head.regression.bias', 'esm.contact_head.regression.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [ ]:
import pandas as pd

model_names = ['facebook/esm2_t12_35M_UR50D', 'facebook/esm2_t30_150M_UR50D']
dataset_names = ['fbp-79k', 'scramm-480k', 'adhesin-3M']

performance_results = pd.DataFrame(index=model_names, columns=dataset_names)

In [21]:
performance_results

,fbp-79k,scramm-480k,adhesin-3M
facebook/esm2_t6_8M_UR50D,NaN,NaN,NaN
facebook/esm2_t12_35M_UR50D,NaN,NaN,NaN
facebook/esm2_t30_150M_UR50D,NaN,NaN,NaN


In [2]:
from datasets import load_from_disk

tokenized_dataset = load_from_disk('../../data/processed/mlm_tokenized')

In [3]:
# Create a subset temporarily
tokenized_dataset = tokenized_dataset['train'].train_test_split(train_size=90, test_size=10, shuffle=True)
tokenized_dataset

DatasetDict({
    train: Dataset({
        features: ['input_ids', 'attention_mask'],
        num_rows: 90
    })
    test: Dataset({
        features: ['input_ids', 'attention_mask'],
        num_rows: 10
    })
})

In [4]:
import torch

def compute_metrics(eval_pred):
    # Convert Hugging Face outputs to PyTorch tensors
    logits, labels = eval_pred
    logits = torch.tensor(logits)  # Shape: (batch_size, seq_len, vocab_size=33)
    labels = torch.tensor(labels)  # Shape: (batch_size, seq_len)
    
    # 1. Identify masked tokens (where labels != -100)
    mask = labels != -100  # Boolean mask for prediction positions
    num_masked_tokens = mask.sum().item()  # Count masked tokens
    
    # Handle edge case: no masked tokens
    if num_masked_tokens == 0:
        return {'pppl_global': float('nan')}

    # 2. Compute log probabilities
    log_probs = torch.nn.functional.log_softmax(logits, dim=-1)  # Shape: (batch_size, seq_len, vocab_size)
    
    # 3. Create safe labels: replace -100 with 0 (to avoid out-of-bounds errors)
    safe_labels = torch.where(labels == -100, 0, labels).long()
    
    # 4. Gather true token log probabilities
    token_log_probs = torch.gather(  # Gather along vocabulary dimension
        input=log_probs, 
        dim=-1, 
        index=safe_labels.unsqueeze(-1)  # Add extra dim for gathering
    ).squeeze(-1)  # Shape: (batch_size, seq_len)
    
    # 5. Compute NLL only for masked tokens
    nll = -token_log_probs[mask]  # Shape: (num_masked_tokens,)
    
    # 6. Calculate pseudo-perplexity
    avg_nll = nll.mean()  # Average NLL over masked tokens
    pppl_global = torch.exp(avg_nll).item()  # exp(avg_nll)
    
    return {'pppl_global': pppl_global}

In [12]:
from transformers import TrainingArguments, Trainer, DataCollatorForLanguageModeling

tokenizer.pad_token = tokenizer.eos_token

data_collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm_probability=0.15)

mlm_ouptut_dir = '../../outputs/mlm'
model_id = 'SpySeer-8M'

training_args = TrainingArguments(
    output_dir=f'{mlm_ouptut_dir}/{model_id}',
    num_train_epochs=2,
    save_strategy='epoch',
    save_total_limit=5,
    logging_strategy='steps',
    logging_steps=10,
    eval_strategy='epoch',
    learning_rate=4e-4,
    weight_decay=0.01,
    per_device_eval_batch_size=4,
    per_device_train_batch_size=4,
    push_to_hub=True,
    hub_model_id=model_id,
    report_to='tensorboard',
    load_best_model_at_end=True,
    metric_for_best_model='eval_pppl_global'
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset['train'],
    eval_dataset=tokenized_dataset['test'],
    data_collator=data_collator,
    processing_class=tokenizer,
    compute_metrics=compute_metrics
)

In [13]:
trainer.train()

Epoch,Training Loss,Validation Loss,Pppl Global
1,1.866800,2.413736,10.903427
2,2.168500,2.412543,11.180655


There were missing keys in the checkpoint model loaded: ['lm_head.decoder.weight'].


TrainOutput(global_step=46, training_loss=1.919856299524722, metrics={'train_runtime': 5.7171, 'train_samples_per_second': 31.484, 'train_steps_per_second': 8.046, 'total_flos': 6342478187472.0, 'train_loss': 1.919856299524722, 'epoch': 2.0})

In [14]:
trainer.state.best_model_checkpoint

'../../outputs/mlm/SpySeer-8M\\checkpoint-46'

In [17]:
best_checkpoint = trainer.state.best_model_checkpoint
model = EsmForMaskedLM.from_pretrained(best_checkpoint)

In [18]:
model

EsmForMaskedLM(
  (esm): EsmModel(
    (embeddings): EsmEmbeddings(
      (word_embeddings): Embedding(33, 320, padding_idx=1)
      (dropout): Dropout(p=0.0, inplace=False)
    )
    (encoder): EsmEncoder(
      (layer): ModuleList(
        (0-5): 6 x EsmLayer(
          (attention): EsmAttention(
            (self): EsmSelfAttention(
              (query): Linear(in_features=320, out_features=320, bias=True)
              (key): Linear(in_features=320, out_features=320, bias=True)
              (value): Linear(in_features=320, out_features=320, bias=True)
              (dropout): Dropout(p=0.0, inplace=False)
              (rotary_embeddings): RotaryEmbedding()
            )
            (output): EsmSelfOutput(
              (dense): Linear(in_features=320, out_features=320, bias=True)
              (dropout): Dropout(p=0.0, inplace=False)
            )
            (LayerNorm): LayerNorm((320,), eps=1e-05, elementwise_affine=True)
          )
          (intermediate): EsmIntermediate